# RolloTree: sklearn Integration & Model Persistence

This notebook covers:
1. `get_params()` / `set_params()` — sklearn estimator protocol
2. `cross_val_score` — cross-validation
3. `GridSearchCV` — hyperparameter tuning
4. `Pipeline` — preprocessing + classification
5. Model persistence — `save()` / `load()` and pickle

In [1]:
import pandas as pd
import numpy as np
import os
import tempfile
import pickle
from rollotree import RollingOCT

In [2]:
# Load and prepare data
train = pd.read_csv("../rollotree/data/train.csv")
test = pd.read_csv("../rollotree/data/test.csv")

X_train = train.drop("y", axis=1)
y_train = train["y"]
X_test = test.drop("y", axis=1)
y_test = test["y"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {sorted(y_train.unique())}")

Train: (160, 130), Test: (18, 130)
Classes: [1, 2, 3]


## 1. sklearn Estimator Protocol

`RollingOCT` implements `get_params()` and `set_params()`, the interface sklearn uses for `clone()`, `GridSearchCV`, and `Pipeline`.

In [3]:
model = RollingOCT(depth=3, criterion="gini", solver="highs", n_jobs=2)

# get_params returns all __init__ parameters
params = model.get_params()
print("All parameters:")
for k, v in sorted(params.items()):
    print(f"  {k:20s} = {v}")

All parameters:
  big_m                = 99
  criterion            = gini
  depth                = 3
  log_to_console       = False
  min_samples_leaf     = 1
  min_samples_split    = 2
  mip_gap              = None
  n_jobs               = 2
  solver               = highs
  time_limit           = 1800


In [4]:
# set_params modifies parameters and returns self (for chaining)
model.set_params(depth=4, criterion="misclassification")
print(f"Updated depth: {model.depth}")
print(f"Updated criterion: {model.criterion}")

Updated depth: 4
Updated criterion: misclassification


In [5]:
# Clone compatibility: create an identical unfitted copy
original = RollingOCT(depth=3, n_jobs=2)
cloned = RollingOCT(**original.get_params())
print(f"Clone params match: {cloned.get_params() == original.get_params()}")

Clone params match: True


## 2. Cross-Validation

Since RolloTree follows the sklearn API, you can use `cross_val_score` directly.

**Note:** Since RolloTree requires binary (0/1) features, the data must be pre-binarized.

In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Combine train+test for cross-validation
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)

# Use StratifiedKFold to ensure each fold has all classes
model = RollingOCT(depth=2, solver="highs")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_all, y_all, cv=cv, scoring="accuracy")

print(f"3-Fold Stratified CV scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

3-Fold Stratified CV scores: [0.58333333 0.49152542 0.59322034]
Mean accuracy: 0.556 (+/- 0.046)


## 3. Grid Search

Use `GridSearchCV` to find the best `depth` and `criterion`.

In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "depth": [2, 3],
    "criterion": ["gini", "misclassification"],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(
    RollingOCT(solver="highs"),
    param_grid,
    cv=cv,
    scoring="accuracy",
    refit=True,
)
grid.fit(X_all, y_all)

print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.3f}")
print(f"\nAll results:")

results_df = pd.DataFrame(grid.cv_results_)[
    ["param_depth", "param_criterion", "mean_test_score", "std_test_score", "mean_fit_time"]
]
results_df

Best params: {'criterion': 'gini', 'depth': 3}
Best CV score: 0.640

All results:


,param_depth,param_criterion,mean_test_score,std_test_score,mean_fit_time
0,2,gini,0.556026,0.045787,1.502064
1,3,gini,0.640019,0.054638,2.893824
2,2,misclassification,0.505179,0.065200,1.378767
3,3,misclassification,0.634557,0.037163,2.985048


In [8]:
# The best model is already refit on the full training data
best_model = grid.best_estimator_
print(f"Best model: {best_model.get_params()}")
print(f"Feature importances (top 5):")

importances = best_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:5]
for i in top_idx:
    print(f"  {X_all.columns[i]:20s}  {importances[i]:.4f}")

Best model: {'depth': 3, 'criterion': 'gini', 'solver': 'highs', 'time_limit': 1800, 'mip_gap': None, 'big_m': 99, 'log_to_console': False, 'min_samples_split': 2, 'min_samples_leaf': 1, 'n_jobs': 1}
Feature importances (top 5):
  91                    0.2000
  62                    0.2000
  61                    0.2000
  31                    0.2000
  21                    0.2000


## 4. Pipeline

RolloTree can be used inside sklearn `Pipeline`. Since it requires binary features, place any binarization step before it.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Identity transform since our data is already binary
pipe = Pipeline([
    ("identity", FunctionTransformer()),  # placeholder for binarization
    ("oct", RollingOCT(depth=2, solver="highs")),
])

pipe.fit(X_train, y_train)
print(f"Pipeline test accuracy: {pipe.score(X_test, y_test):.3f}")
print(f"Pipeline predict_proba shape: {pipe.predict_proba(X_test).shape}")

Pipeline test accuracy: 0.611
Pipeline predict_proba shape: (18, 3)


## 5. Model Persistence

### 5.1 Built-in `save()` / `load()`

Uses `joblib` under the hood for efficient serialization.

In [10]:
# Train a model
model = RollingOCT(depth=3, solver="highs")
model.fit(X_train, y_train)
preds_before = model.predict(X_test)
proba_before = model.predict_proba(X_test)

print(f"Accuracy before save: {model.score(X_test, y_test):.3f}")

Accuracy before save: 0.778


In [11]:
# Save and load
with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
    path = f.name

model.save(path)
print(f"Saved to: {path}")
print(f"File size: {os.path.getsize(path) / 1024:.1f} KB")

loaded = RollingOCT.load(path)
preds_after = loaded.predict(X_test)
proba_after = loaded.predict_proba(X_test)

print(f"Accuracy after load:  {loaded.score(X_test, y_test):.3f}")
print(f"Predictions match: {np.array_equal(preds_before, preds_after)}")
print(f"Probabilities match: {np.array_equal(proba_before, proba_after)}")
print(f"Params match: {loaded.get_params() == model.get_params()}")

os.unlink(path)

Saved to: /var/folders/cq/qzcx1_8n19v_1twt74dtbzhxkyzxlm/T/tmp415uo1dq.joblib
File size: 9.2 KB
Accuracy after load:  0.778
Predictions match: True
Probabilities match: True
Params match: True


### 5.2 Pickle

Standard Python `pickle` also works.

In [12]:
data = pickle.dumps(model)
loaded_pkl = pickle.loads(data)

print(f"Pickle size: {len(data) / 1024:.1f} KB")
print(f"Predictions match: {np.array_equal(model.predict(X_test), loaded_pkl.predict(X_test))}")

Pickle size: 8.9 KB
Predictions match: True


## 6. Input Validation

RolloTree validates inputs at `fit()` time and raises clear error messages.

In [13]:
# Non-binary features raise ValueError
try:
    bad_X = np.array([[0, 1, 2], [1, 0, 3]])
    bad_y = np.array([1, 2])
    RollingOCT(depth=2).fit(bad_X, bad_y)
except ValueError as e:
    print(f"Caught: {e}")

Caught: X must contain only binary (0/1) values. Found non-binary values: [2, 3]. Use rollotree.preprocessing.helpers.make_data_binary() to binarize your data.


In [14]:
# Single class raises ValueError
try:
    X_ok = np.array([[0, 1], [1, 0], [0, 0]])
    y_bad = np.array([1, 1, 1])
    RollingOCT(depth=2).fit(X_ok, y_bad)
except ValueError as e:
    print(f"Caught: {e}")

Caught: y must have at least 2 classes, found 1.


In [15]:
# Invalid set_params raises ValueError
try:
    RollingOCT().set_params(nonexistent_param=42)
except ValueError as e:
    print(f"Caught: {e}")

Caught: Invalid parameter 'nonexistent_param' for RollingOCT. Valid parameters: ['depth', 'criterion', 'solver', 'time_limit', 'mip_gap', 'big_m', 'log_to_console', 'min_samples_split', 'min_samples_leaf', 'n_jobs']


In [16]:
# Calling predict/predict_proba before fit raises RuntimeError
try:
    RollingOCT().predict(np.zeros((5, 3)))
except RuntimeError as e:
    print(f"Caught: {e}")

Caught: Model has not been fitted. Call fit() first.


In [17]:
# Feature names are stored from DataFrames
model_fn = RollingOCT(depth=2)
model_fn.fit(X_train, y_train)
print(f"n_features_in_: {model_fn.n_features_in_}")
print(f"feature_names_in_ (first 10): {model_fn.feature_names_in_[:10]}")

n_features_in_: 130
feature_names_in_ (first 10): ['1' '2' '3' '4' '5' '6' '7' '8' '9' '10']


## Summary

| Feature | Code |
|---------|------|
| Get all params | `model.get_params()` |
| Set params | `model.set_params(depth=4)` |
| Cross-validate | `cross_val_score(model, X, y, cv=5)` |
| Grid search | `GridSearchCV(model, param_grid).fit(X, y)` |
| Pipeline | `Pipeline([("oct", RollingOCT())])` |
| Save model | `model.save("model.joblib")` |
| Load model | `RollingOCT.load("model.joblib")` |
| Pickle | `pickle.dumps(model)` / `pickle.loads(data)` |

See **01_quickstart.ipynb** for basics, **02_visualization.ipynb** for tree visualization.